# บท 03 · ตรวจข้อมูลราคาก่อนใช้

ข้อมูลจำลองเจ็ดแถวมีทั้งลำดับเวลาสลับ แถวซ้ำ ราคาหาย และ OHLCV ผิด ใช้เพื่อฝึกแยกปัญหาโดยไม่ทำลายหลักฐานต้นฉบับ ไม่มีคำขอเครือข่าย

เวลา 5 มกราคม 2026 เป็นเวลาจำลอง ไม่อ้างว่าเป็น session ของตลาดจริง แท่งหนึ่งนาทีมี timestamp สิ้นสุดแท่งตามสัญญาข้อมูลที่สร้างขึ้นเอง


## 1. สภาพแวดล้อม


In [1]:
import platform
import numpy as np
import pandas as pd
print({"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__})

from io import StringIO
import hashlib


{'python': '3.12.14', 'numpy': '2.5.3', 'pandas': '2.3.2'}


## 2. อ่านข้อมูลดิบเป็นข้อความก่อน

คงค่าที่รับมาให้ตรวจย้อนหลังได้ ข้อมูล CSV อยู่ใน Notebook จึงไม่ต้องดาวน์โหลดไฟล์อื่น


In [2]:
RAW_CSV = """time,open,high,low,close,volume
2026-01-05T14:33:00Z,101,104,100,103,130
2026-01-05T14:31:00Z,100,102,99,101,100
2026-01-05T14:32:00Z,101,103,100,102,110
2026-01-05T14:32:00Z,101,103,100,102,110
2026-01-05T14:34:00Z,103,104,101,,140
2026-01-05T14:35:00Z,102,103,100,101,150
2026-01-05T14:36:00Z,101,99,100,102,-1
"""
FIELDS = ["open", "high", "low", "close", "volume"]
raw = pd.read_csv(StringIO(RAW_CSV), dtype=str, keep_default_na=False)
original = raw.copy(deep=True)
print(raw.to_string(index=False))


                time open high low close volume
2026-01-05T14:33:00Z  101  104 100   103    130
2026-01-05T14:31:00Z  100  102  99   101    100
2026-01-05T14:32:00Z  101  103 100   102    110
2026-01-05T14:32:00Z  101  103 100   102    110
2026-01-05T14:34:00Z  103  104 101          140
2026-01-05T14:35:00Z  102  103 100   101    150
2026-01-05T14:36:00Z  101   99 100   102     -1


## 3. แปลง ตรวจ และแยกแถวที่ใช้ไม่ได้

แถวซ้ำทุกฟิลด์ตัดออกหนึ่งสำเนาพร้อมนับจำนวน หาก timestamp เดียวกันมีค่าต่างกัน จะแยกทุกแถวที่ขัดแย้ง ไม่มีการเลือกข้อมูลที่ดูสะดวกกว่า


In [3]:
def validate_bars(raw):
    """Keep raw input untouched; quarantine bad or conflicting rows, never fill prices."""
    required = ["time"] + FIELDS
    if not set(required).issubset(raw.columns):
        raise ValueError("missing required columns")
    exact_duplicate = raw.duplicated(subset=required, keep="first")
    work = raw.loc[~exact_duplicate, required].copy()
    # utc=True is valid here because the fixture declares every timestamp in UTC.
    work["time"] = pd.to_datetime(work["time"], format="ISO8601", utc=True, errors="coerce")
    for name in FIELDS:
        work[name] = pd.to_numeric(work[name], errors="coerce")
    price = work[["open", "high", "low", "close"]]
    finite = np.isfinite(work[FIELDS]).all(axis=1)
    flags = pd.DataFrame({
        "invalid_time": work["time"].isna(),
        "missing_or_nonfinite": ~finite,
        "nonpositive_price": (price <= 0).any(axis=1),
        "negative_volume": work["volume"] < 0,
        "invalid_ohlc": (work["high"] < price.max(axis=1)) | (work["low"] > price.min(axis=1)),
        "conflicting_timestamp": work["time"].notna() & work["time"].duplicated(keep=False),
    }, index=work.index)
    bad = flags.any(axis=1)
    quarantine = work.loc[bad].copy()
    quarantine["reason"] = flags.loc[bad].apply(lambda row: ";".join(row.index[row]), axis=1)
    clean = work.loc[~bad].sort_values("time").set_index("time")
    report = {"raw_rows": len(raw), "exact_duplicates_removed": int(exact_duplicate.sum()),
              "quarantined_rows": len(quarantine), "accepted_rows": len(clean)}
    assert sum(report[k] for k in ["exact_duplicates_removed", "quarantined_rows", "accepted_rows"]) == len(raw)
    return clean, quarantine, report
print("Validator ready; raw input is never changed.")


Validator ready; raw input is never changed.


## 4. ผลการตรวจต้องอธิบายจำนวนแถวได้

รายงานควรเป็น 7 = 1 แถวซ้ำ + 2 แถวแยก + 4 แถวผ่าน การแยกแถวไม่ใช่การแก้ช่องว่างของตารางเวลา


In [4]:
clean, quarantine, report = validate_bars(raw)
assert report == {"raw_rows": 7, "exact_duplicates_removed": 1, "quarantined_rows": 2, "accepted_rows": 4}
pd.testing.assert_frame_equal(raw, original)
assert clean.index.is_monotonic_increasing and clean.index.is_unique
print(report)
print(clean.to_string())
print(quarantine[["time", "reason"]].to_string(index=False))


{'raw_rows': 7, 'exact_duplicates_removed': 1, 'quarantined_rows': 2, 'accepted_rows': 4}
                           open  high  low  close  volume
time                                                     
2026-01-05 14:31:00+00:00   100   102   99  101.0     100
2026-01-05 14:32:00+00:00   101   103  100  102.0     110
2026-01-05 14:33:00+00:00   101   104  100  103.0     130
2026-01-05 14:35:00+00:00   102   103  100  101.0     150
                     time                       reason
2026-01-05 14:34:00+00:00         missing_or_nonfinite
2026-01-05 14:36:00+00:00 negative_volume;invalid_ohlc


## 5. เปิดให้เห็นช่องว่างในตาราง

เรากำหนดตารางเวลาจำลอง 14:31–14:35 UTC ไว้ก่อน เมื่อไม่มีแท่ง 14:34 ผลตอบแทนหนึ่งนาทีที่ 14:34 และ 14:35 ต้องไม่ถูกสร้างขึ้นจากการเติมราคา


In [5]:
expected = pd.date_range("2026-01-05T14:31:00Z", periods=5, freq="min")
missing = expected.difference(clean.index)
unexpected = clean.index.difference(expected)
regular = clean.reindex(expected)
regular["return"] = regular["close"].pct_change(fill_method=None)
assert len(missing) == 1 and len(unexpected) == 0
assert regular["return"].iloc[-2:].isna().all()
print("Missing:", missing.astype(str).tolist())
print(regular[["close", "return"]].round(6).to_string())
print(f"Observed two-minute change 14:33 to 14:35: {(101/103-1):.6%}")


Missing: ['2026-01-05 14:34:00+00:00']
                           close    return
2026-01-05 14:31:00+00:00  101.0       NaN
2026-01-05 14:32:00+00:00  102.0  0.009901
2026-01-05 14:33:00+00:00  103.0  0.009804
2026-01-05 14:34:00+00:00    NaN       NaN
2026-01-05 14:35:00+00:00  101.0       NaN
Observed two-minute change 14:33 to 14:35: -1.941748%


## 6. แบบฝึกหัด: ค่าไม่ตรงกันที่เวลาเดียวกัน

เพิ่มแถว 14:31 ที่ close เป็น 100 แทน 101 แล้วตรวจว่าทั้งคู่ถูกแยก ไม่ตัดสินว่าแถวล่าสุดถูกเสมอ


In [6]:
conflict = raw.iloc[[1]].copy()
conflict["close"] = "100"
raw_conflict = pd.concat([raw, conflict], ignore_index=True)
clean2, quarantine2, report2 = validate_bars(raw_conflict)
assert report2 == {"raw_rows": 8, "exact_duplicates_removed": 1, "quarantined_rows": 4, "accepted_rows": 3}
assert quarantine2["reason"].str.contains("conflicting_timestamp").sum() == 2
print(report2)
print(quarantine2[["time", "close", "reason"]].to_string(index=False))


{'raw_rows': 8, 'exact_duplicates_removed': 1, 'quarantined_rows': 4, 'accepted_rows': 3}
                     time  close                       reason
2026-01-05 14:31:00+00:00  101.0        conflicting_timestamp
2026-01-05 14:34:00+00:00    NaN         missing_or_nonfinite
2026-01-05 14:36:00+00:00  102.0 negative_volume;invalid_ohlc
2026-01-05 14:31:00+00:00  100.0        conflicting_timestamp


## 7. เก็บหลักฐาน และตรวจการอ่านกลับ

เรา serialize ข้อมูลสะอาดเป็น CSV ในหน่วยความจำ แล้วอ่านกลับเพื่อตรวจเวลาและค่า ไม่เขียนทับ raw file ค่า hash อ้างถึงข้อความ CSV ดิบตามที่ประกาศไว้ในเซลล์


In [7]:
checksum = hashlib.sha256(RAW_CSV.encode()).hexdigest()
assert checksum == "ca6626c9b2e137303e980a81c8ffb6b4d4109a41f9d7247707ce40eef71bec7e"
csv_text = clean.to_csv()
restored = pd.read_csv(StringIO(csv_text))
restored["time"] = pd.to_datetime(restored["time"], utc=True)
restored = restored.set_index("time")
pd.testing.assert_frame_equal(clean, restored, check_freq=False)
pd.testing.assert_frame_equal(raw, original)
print("Raw SHA256:", checksum)
print("CSV round trip preserved all accepted values and UTC timestamps.")


Raw SHA256: ca6626c9b2e137303e980a81c8ffb6b4d4109a41f9d7247707ce40eef71bec7e
CSV round trip preserved all accepted values and UTC timestamps.


## 8. ตัวอย่าง split ที่แยกจากผลตอบแทน

นี่เป็นตัวอย่างเชิงบัญชี 2 ต่อ 1 ไม่ใช่ corporate action ของบริษัทจริง ราคาหารสองและจำนวนหุ้นคูณสองทำให้มูลค่าถือครองเท่าเดิมก่อนผลตลาดอื่น ๆ


In [8]:
before_shares, before_price = 10, 100
ratio = 2
after_shares, after_price = before_shares * ratio, before_price / ratio
assert before_shares * before_price == after_shares * after_price
print({"before_value": before_shares * before_price, "after_value": after_shares * after_price})
print(f"Naive unadjusted price change: {after_price / before_price - 1:.0%}")


{'before_value': 1000, 'after_value': 1000.0}
Naive unadjusted price change: -50%


## แหล่งอ้างอิงและสิ่งที่ยังไม่พิสูจน์

- [pandas 2.3 to_datetime](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.to_datetime.html)
- [pandas 2.3 reindex](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.DataFrame.reindex.html)
- [Investor.gov Stock Split](https://www.investor.gov/introduction-investing/investing-basics/glossary/stock-split)
- Hilpisch บท 3 หน้าเล่ม 45–47 และ 49–50 (PDF 65–67, 69–70): กรอบนำเข้าและจัดการข้อมูล; ตัวอย่างนี้เขียนใหม่

ตรวจแหล่งอ้างอิง 11 กันยายน 2026 การผ่าน validation ไม่รับรองวิธีปรับราคา ปฏิทินตลาด สิทธิ์ข้อมูล หรือความถูกต้องแบบ point-in-time ต้องตรวจแหล่งจริงเพิ่มก่อนใช้ backtest ตลาดจริง
